# TODO
- Concatentate tokens, xs, and ys into one row (many rows)
- Train heuristic function and it will save hueristic network to .pt file
>> deepxube train --domain grid.7 --heur resnet_fc.100H_2B_bn --heur_type V --pathfind graph_v --step_max 100 --up_itrs 100 --search_itrs 20 --backup -1 --procs 1 --batch_size 50 --max_itrs 5000 --dir dummy/

- Solve with trained heuristic
>> deepxube solve --domain grid.7 --heur resnet_fc.100H_2B_bn --heur_file dummy/heur.pt --heur_type V --pathfind graph_v.1B_1.0W --file valid.pkl --results results_trained_ex/ --redo

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np

from sympy import *
from sympy.abc import x

from transformers import BertTokenizer, BertTokenizerFast

from deepxube.domains.symbolic_regression import SymbolicState, SymbolicGoal, SymbolicRegression

## Tokenizer setup

In [ ]:
def normalize(expr):
    import re
    return " ".join(re.findall(r'\d+|[a-zA-Z]+|[\+\-\*/\^\=\(\)]', expr))

In [ ]:
tokenizer = BertTokenizer.from_pretrained(
    './', 
    lowercase=True,
)
# tokenizer.vocab

In [ ]:
# Why do I need to pass the attention mask in this case? It's 1 for "real" tokens and 0 for 
encoding = tokenizer(
    'sin(x) 1 2 3',
    padding='max_length',
    return_tensors='np',
    max_length=128
)

np.concatenate((encoding['input_ids'], encoding['attention_mask']), axis=None)

## Testing to_np

In [ ]:
def _state_to_np(state: SymbolicState, tokenizer: BertTokenizer) -> np.ndarray:
    encoding = tokenizer(
        str(state.expr),
        padding='max_length',
        return_tensors='np',
        max_length=64
    )
    return np.concatenate((encoding['input_ids'], encoding['attention_mask']), axis=None)

def to_np(states: list[SymbolicState], goals: list[SymbolicGoal]) -> list[list[np.ndarray]]:
    # Each row should be a problem instance
    # should each row (i.e. the token array) be padded to the same length? Yes, I think
    # TODO: Concatenate each row of the to_np output
    tokenizer = BertTokenizer.from_pretrained('./', lowercase=True)
    tokens = [_state_to_np(state, tokenizer) for state in states]
    return [[ts, g.xs, g.ys] for ts, g in zip(tokens, goals)]

In [ ]:
start_state = SymbolicState(2*x + x**2)
_state_to_np(start_state, tokenizer)

In [ ]:
# Make first state and goal pair
start_state1 = SymbolicState(2*x + x**2)
goal1 = SymbolicGoal(
    xs=np.array([0, 1, 2, 3, 4]),
    ys=np.array([0, 1, 4, 9, 16]),
    tolerance=0
)

# Second state and goal pair
start_state2 = SymbolicState(3*x + 1 + sin(-x))
goal2 = SymbolicGoal(
    xs=np.array([0, 1, 2, 3, 4]),
    ys=np.array([0, 1, 8, 27, 64]),
    tolerance=0
)

# Convert to NP
out = to_np(
    [start_state1, start_state2], 
    [goal1, goal2]
)

In [ ]:
       # [
        #   [token_array1, goal_x_array1, goal_y_array1],
        #   [token_array2, goal_x_array2, goal_y_array2],
        #   etc
        # ]
out